# Lab 19: DDP and MLflow

**Module 19** | Read `notes/19-ddp-mlops.pdf` before running this notebook.

Train a small CNN on MNIST while logging parameters, metrics, and the model artifact to MLflow.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


# Lab 19: DDP and MLflow

**Module 19** | Read `notes/19-ddp-mlops.pdf` before running this notebook.

Train a small CNN on MNIST while logging parameters, metrics, and the model artifact to MLflow.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Experiment Tracking with MLflow

MLOps workflows need reproducible records of **what was trained, with which hyperparameters, and how well it performed**.
[MLflow Tracking](https://mlflow.org/docs/latest/tracking.html) logs params, metrics, and artifacts (including PyTorch models) to a local or remote store.

This lab trains a **small CNN on MNIST** for a few epochs and logs:

- Hyperparameters (`epochs`, `learning_rate`, `batch_size`, `device`)
- Per-epoch training loss and final test accuracy
- The trained model via `mlflow.pytorch.log_model`

The tracking URI comes from the environment (`MLFLOW_TRACKING_URI`), set automatically by `runtime_env.configure_runtime()` to the project's `mlruns/` folder.
Run `mlflow ui` (started by `env.ps1`) to browse logged runs in the browser.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

data_root = ROOT / "datasets" / "mnist"
train_ds = datasets.MNIST(str(data_root), train=True, download=True, transform=transform)
test_ds = datasets.MNIST(str(data_root), train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)

epochs = 2
lr = 1e-3

tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", str(ROOT / "mlruns"))
mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")


In [ ]:
with mlflow.start_run(run_name="lab19_mnist_cnn") as run:
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("learning_rate", lr)
    mlflow.log_param("batch_size", 128)
    mlflow.log_param("device", str(device))
    mlflow.log_param("architecture", "SmallCNN")

    model = SmallCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        mlflow.log_metric("train_loss", epoch_loss, step=epoch)
        print(f"Epoch {epoch + 1}/{epochs}, train loss: {epoch_loss:.4f}")

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    test_acc = correct / total
    mlflow.log_metric("test_accuracy", test_acc)

    mlflow.pytorch.log_model(model, artifact_path="mnist_cnn")

    print(f"Test accuracy: {test_acc:.4f}")
    print(f"MLflow run ID: {run.info.run_id}")
    print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
    print("Open MLflow UI (http://127.0.0.1:5000) to inspect params, metrics, and the saved model.")


## Optional: Distributed Data Parallel (DDP)

For multi-GPU training, PyTorch **DistributedDataParallel** wraps the model and synchronizes gradients across processes launched with `torchrun`:

```bash
torchrun --nproc_per_node=2 train_script.py
```

Each process runs on one GPU; only rank 0 should call `mlflow.log_*` to avoid duplicate runs.
This lab uses single-process training so it runs on any machine; the MLflow logging pattern is identical in DDP, only the training loop launches differ.

**Deliverable:** note the printed **run ID** and confirm the run appears in the MLflow UI with `test_accuracy` and a `mnist_cnn` model artifact.
